# Saved GAN generator: BatchNorm mode diagnostic

Executed 10 November 2026 on Kaggle CPU, TensorFlow 2.21.0 / Python 3.13.18. The archived pilot used TensorFlow 2.20.0; runtime drift was explicitly enabled and reported. This is a diagnostic, not a new training run or exact runtime reproduction.

Same saved generator, identical noise and labels; compare inference and training modes. Model state is restored after calls. No optimizer updates and no training/validation/test image datasets are read. Archived synthetic grids are used only for a replay check.

Both modes still show collapse and checkerboard artifacts. The results do not prove that changing BatchNorm mode will fix training. See [results](../docs/gan-mode-diagnostic.md).

Image display payloads are omitted from this GitHub copy for readability; source and numerical outputs are retained. All six grids and full executed display outputs are preserved in [Kaggle notebook 10](https://www.kaggle.com/code/amirsalahk/10-isic-gan-batchnorm-diagnostic/edit), Version 1, with output saving selected. Original exported notebook remains unchanged locally. Running again requires a fresh output directory or a fresh session because the script refuses to overwrite existing results.


In [3]:
"""Compare saved generator BatchNorm modes; no optimizer or training-data access.

Run in the original TensorFlow 2.20.0 Kaggle environment:
python diagnose_cgan_modes.py --model /path/generator_step05000.keras \
    --output /kaggle/working/cgan_mode_diagnostic
"""
import argparse
import hashlib
import json
import platform
from pathlib import Path


def main():
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument('--model', required=True, type=Path)
    parser.add_argument('--output', required=True, type=Path)
    parser.add_argument('--allow-runtime-drift', action='store_true',
                        help='Explicit exploratory comparison in a different TensorFlow version')
    args = parser.parse_args()
    import numpy as np
    import tensorflow as tf
    from PIL import Image

    assert tf.__version__ == '2.20.0' or args.allow_runtime_drift, 'Use the pilot runtime or explicitly allow runtime drift'
    assert not args.output.exists(), 'Choose a fresh output directory'
    model = tf.keras.models.load_model(args.model, compile=False)
    original = model.get_weights()
    digest = lambda weights: hashlib.sha256(b''.join(a.tobytes() for a in weights)).hexdigest()
    before = digest(original)
    noise = np.random.default_rng(4142).normal(size=(32, 128)).astype('float32')
    cases = {'all_benign': [0] * 32, 'all_malignant': [1] * 32,
             'mixed': [0] * 16 + [1] * 16}
    arrays, results = {}, {}
    try:
        for name, labels in cases.items():
            outputs = {}
            for mode in [False, True]:
                # Reset all state before each call: training=True updates BN statistics.
                model.set_weights(original)
                a = model([tf.constant(noise), tf.constant(labels, tf.int32)],
                          training=mode).numpy()
                assert a.shape == (32, 128, 128, 3) and np.isfinite(a).all()
                outputs[mode] = (a + 1) / 2  # Metrics use floating pixels on [0, 1].
                arrays[f'{name}_{"training" if mode else "inference"}'] = outputs[mode]
            results[name] = {
                'mode_mean_absolute_difference': float(np.abs(outputs[True] - outputs[False]).mean()),
                'inference_mean_pixel_sd': float(outputs[False].std(axis=0).mean()),
                'training_mean_pixel_sd': float(outputs[True].std(axis=0).mean()),
            }
    finally:
        model.set_weights(original)
    assert digest(model.get_weights()) == before, 'Model state was not restored'
    results['same_noise_class_change_mae'] = {
        mode: float(np.abs(arrays[f'all_benign_{mode}'] - arrays[f'all_malignant_{mode}']).mean())
        for mode in ['training', 'inference']
    }
    args.output.mkdir(parents=True)
    for name, a in arrays.items():
        canvas = Image.new('RGB', (512, 512))
        for j, pixels in enumerate(a[:16]):
            canvas.paste(Image.fromarray(np.clip(pixels * 255, 0, 255).round().astype('uint8')),
                         ((j % 4) * 128, (j // 4) * 128))
        canvas.save(args.output / f'{name}.png')
    report = {'tensorflow': tf.__version__, 'python': platform.python_version(),
              'pilot_tensorflow': '2.20.0', 'runtime_matches_pilot': tf.__version__ == '2.20.0',
              'runtime_drift_explicitly_allowed': args.allow_runtime_drift,
              'model_file': str(args.model),
              'model_file_sha256': hashlib.sha256(args.model.read_bytes()).hexdigest(),
              'noise_seed': 4142, 'batch_size': 32, 'model_state_restored': True,
              'optimizer_updates': 0, 'results': results,
              'limitations': 'Descriptive only; no pass threshold, causal proof, clinical validity, '
                             'or memorization assessment. Training-mode output depends on batch composition.'}
    (args.output / 'mode_diagnostic.json').write_text(json.dumps(report, indent=2))
    print(json.dumps(report, indent=2))


# Definitions only; explicit invocation below.

import sys
models = sorted(Path("/kaggle/input").rglob("generator_step05000.keras"))
assert len(models) == 1
assert hashlib.sha256(models[0].read_bytes()).hexdigest() == "a67c5a17e94f8e07540658818c071ddf9481203321fa4b7f279751097a88fdbb"
sys.argv = ["diagnose_cgan_modes.py", "--model", str(models[0]), "--output", "/kaggle/working/cgan_mode_diagnostic", "--allow-runtime-drift"]
main()
import shutil
shutil.make_archive("/kaggle/working/cgan_mode_diagnostic", "zip", "/kaggle/working/cgan_mode_diagnostic")
print("MODE_DIAGNOSTIC_COMPLETE")


E0000 00:00:1794344467.785563      49 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


{
  "tensorflow": "2.21.0",
  "python": "3.13.18",
  "pilot_tensorflow": "2.20.0",
  "runtime_matches_pilot": false,
  "runtime_drift_explicitly_allowed": true,
  "model_file": "/kaggle/input/notebooks/amirsalahk/09-isic-binary-conditional-gan-pilot/binary_cgan_pilot/generator_step05000.keras",
  "model_file_sha256": "a67c5a17e94f8e07540658818c071ddf9481203321fa4b7f279751097a88fdbb",
  "noise_seed": 4142,
  "batch_size": 32,
  "model_state_restored": true,
  "optimizer_updates": 0,
  "results": {
    "all_benign": {
      "mode_mean_absolute_difference": 0.1058327779173851,
      "inference_mean_pixel_sd": 0.03354823961853981,
      "training_mean_pixel_sd": 0.04649041220545769
    },
    "all_malignant": {
      "mode_mean_absolute_difference": 0.10518241673707962,
      "inference_mean_pixel_sd": 0.03405648469924927,
      "training_mean_pixel_sd": 0.047614481300115585
    },
    "mixed": {
      "mode_mean_absolute_difference": 0.1055193543434143,
      "inference_mean_pixel_sd": 0.

In [4]:
from PIL import Image
from IPython.display import display
import numpy as np, json, shutil
from pathlib import Path
folder = Path('/kaggle/working/cgan_mode_diagnostic')
base = models[0].parent
replay = {}
for target, name in [(0,'all_benign'),(1,'all_malignant')]:
    original = np.asarray(Image.open(base / f'grid_class{target}_05000.png').convert('RGB'))
    current = np.asarray(Image.open(folder / f'{name}_inference.png').convert('RGB'))
    if target == 0:
        comparable = current
    else:
        # Original malignant grid uses latent rows 16:32, not 0:16.
        import tensorflow as tf
        replay_model = tf.keras.models.load_model(models[0], compile=False)
        z = np.random.default_rng(4142).normal(size=(32,128)).astype('float32')
        a = replay_model([tf.constant(z),tf.constant([0]*16+[1]*16)],training=False).numpy()[16:]
        canvas = Image.new('RGB',(512,512))
        for j,pixels in enumerate(a):
            canvas.paste(Image.fromarray(np.clip((pixels+1)*127.5,0,255).round().astype('uint8')),((j%4)*128,(j//4)*128))
        comparable = np.asarray(canvas)
    difference = np.abs(original.astype('float32') - comparable.astype('float32'))
    replay[str(target)] = dict(mean_absolute_uint8_difference=float(difference.mean()),max_uint8_difference=float(difference.max()),fraction_channel_values_different=float(np.mean(difference != 0)))
print('ARCHIVED_GRID_REPLAY',json.dumps(replay))
(folder/'runtime_replay.json').write_text(json.dumps(replay,indent=2))
for name in ['all_benign_inference','all_benign_training','all_malignant_inference','all_malignant_training']:
    print(name)
    display(Image.open(folder/f'{name}.png'))
shutil.make_archive('/kaggle/working/cgan_mode_diagnostic','zip',folder)
print('REVIEW_COMPLETE')


ARCHIVED_GRID_REPLAY {"0": {"mean_absolute_uint8_difference": 5.086263172415784e-06, "max_uint8_difference": 1.0, "fraction_channel_values_different": 5.086263020833333e-06}, "1": {"mean_absolute_uint8_difference": 8.900959983293433e-06, "max_uint8_difference": 1.0, "fraction_channel_values_different": 8.900960286458334e-06}}
all_benign_inference


all_benign_training


all_malignant_inference


all_malignant_training


REVIEW_COMPLETE
